# Fine-tune Qwen2.5-VL-3B on the HCFA-1500 dataset (QLoRA)

End-to-end Colab notebook for the `catochris/hcfa-1500` dataset:

1. Load the dataset straight from the Hugging Face Hub.
2. QLoRA fine-tune **Qwen/Qwen2.5-VL-3B-Instruct** (image + prompt → flat JSON).
3. Run inference on the `test` split and score it with **your own `hcfa_eval` harness**
   (per-tier, per-field-class, CER, JSON-validity).

**Tuned to run on a free Colab T4 (15 GB).** The notebook auto-detects bf16 (A100) vs
fp16 (T4) and caps image resolution so it fits. On a bigger GPU just raise `MAX_PIXELS`
and `per_device_train_batch_size` where flagged.

> **Free T4 notes:** T4 is fp16-only (no bf16) and 15 GB, so we cap `MAX_PIXELS`, use
> batch size 1 + grad accumulation, an 8-bit optimizer, and a short prompt. A 3-epoch
> run is ~30–60 min. Free runtimes can disconnect — we save the adapter every epoch and
> you can push it to the Hub.

## 1. Install dependencies

In [ ]:
# Qwen2.5-VL needs transformers>=4.49. trl/peft/bitsandbytes for QLoRA SFT.
%pip install -q -U "transformers>=4.49" "trl>=0.12" "peft>=0.13" accelerate bitsandbytes datasets qwen-vl-utils
# hcfa_eval (+ hcfa_synth schema) so we can score with the repo's harness in-notebook:
%pip install -q "git+https://github.com/chrscato/hcfa-1500-reader.git"

## 2. Check the GPU (and pick fp16 vs bf16)

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> GPU (T4)."
name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()          # False on T4 (Turing), True on A100/L4
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(f"GPU: {name} ({total_gb:.0f} GB) | bf16={BF16} -> compute dtype {DTYPE}")
IS_SMALL_GPU = total_gb < 24                    # T4/16GB path

## 3. Authenticate + load the dataset

`login()` opens a token box — paste a token from https://huggingface.co/settings/tokens
(a **read** token is enough to load; you need **write** later to push the adapter).

In [ ]:
from huggingface_hub import login
login()  # paste your HF token

from datasets import load_dataset

REPO_ID = "catochris/hcfa-1500"
ds = load_dataset(REPO_ID)
print(ds)

In [ ]:
# Peek at one example: an image, the prompt, and the flat-JSON target.
ex = ds["train"][0]
print("columns:", ds["train"].column_names)
print("tier:", ex["tier"], "| sample_id:", ex["sample_id"])
print("image size:", ex["image"].size)
print("target (head):", ex["target"][:300], "...")
ex["image"]

## 4. Prompt mode (parity knob)

The dataset's `prompt` column lists all ~118 schema keys (great for *zero-shot* eval).
For **fine-tuning** the model learns the schema, so a short prompt is cheaper and avoids
overflowing the sequence length on a T4. `get_prompt()` is used in BOTH training and
inference, so whichever mode you pick stays consistent (train/serve parity).

In [ ]:
PROMPT_MODE = "minimal" if IS_SMALL_GPU else "schema"  # "minimal" (short) | "schema" (dataset's full-key prompt)

MINIMAL_PROMPT = (
    "Extract every field from this CMS-1500 (HCFA) claim form as a single flat JSON "
    'object. Use "" for any blank field. Return only the JSON.'
)


def get_prompt(example):
    return MINIMAL_PROMPT if PROMPT_MODE == "minimal" else example["prompt"]


print("PROMPT_MODE =", PROMPT_MODE)

## 5. Load Qwen2.5-VL-3B in 4-bit (QLoRA base)

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# Resolution is the main accuracy lever for ~100 small-text fields, but more pixels =
# more vision tokens = more VRAM. The HCFA forms render at 2550x3300, so we MUST cap.
#   T4 (15 GB):  1024 * 28*28  (~0.8M px, ~256 vision tokens)  <- default
#   A100/L4:     1280 * 28*28+  (raise this first when you have headroom)
# If you OOM on the T4, drop the multiplier to 768 or 512.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = (1024 if IS_SMALL_GPU else 1280) * 28 * 28

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,       # float16 on T4, bfloat16 on A100
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    torch_dtype=DTYPE,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "right"
print("loaded", MODEL_ID, "| max_pixels =", MAX_PIXELS)

## 6. Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False  # required with gradient checkpointing

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Adapt the language tower only; leave the vision encoder frozen (cheaper, stabler).
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 7. Collator: image + prompt → masked-label batch

We build Qwen chat messages on the fly (image inline so `process_vision_info` can find
it), render them with the chat template, then mask the prompt-side and image tokens so
loss is computed only on the target JSON.

In [ ]:
from qwen_vl_utils import process_vision_info

IMAGE_PAD_ID = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")


def to_messages(example, with_target=True):
    msgs = [{
        "role": "user",
        "content": [
            {"type": "image", "image": example["image"]},
            {"type": "text", "text": get_prompt(example)},
        ],
    }]
    if with_target:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": example["target"]}]})
    return msgs


def collate_fn(examples):
    batch_msgs = [to_messages(ex) for ex in examples]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch_msgs]
    images = []
    for m in batch_msgs:
        imgs, _ = process_vision_info(m)
        images.extend(imgs)
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == IMAGE_PAD_ID] = -100  # don't train on vision placeholder tokens
    batch["labels"] = labels
    return batch


# sanity check one batch (also a quick VRAM + sequence-length canary)
_b = collate_fn([ds["train"][0], ds["train"][1]])
print({k: tuple(v.shape) for k, v in _b.items() if hasattr(v, "shape")})
print("seq len:", _b["input_ids"].shape[1], "(must be < max_seq_length or the target truncates)")

## 8. Train (QLoRA SFT)

On a free T4 expect ~30–60 min for 3 epochs. If you hit CUDA OOM: lower `MAX_PIXELS`
(cell 5) and re-run from cell 5, or cut `num_train_epochs`. The adapter is saved every
epoch to `qwen2.5vl-3b-hcfa/`, so a disconnect doesn't cost you everything.

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="qwen2.5vl-3b-hcfa",
    num_train_epochs=3,                 # 404 synthetic examples — watch val loss for overfit
    per_device_train_batch_size=1,      # raise to 2-4 only on >=24 GB GPUs
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=BF16,                          # auto: bf16 on A100, fp16 on T4
    fp16=not BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",           # 8-bit optimizer states — saves VRAM on the T4
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    max_seq_length=4096,                # fits with the minimal prompt; raise to ~6144 for "schema" mode
    dataset_kwargs={"skip_prepare_dataset": True},  # our collator does the work
    remove_unused_columns=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=collate_fn,
    processing_class=processor,
)
trainer.train()

## 9. Save / push the adapter

In [ ]:
ADAPTER_DIR = "qwen2.5vl-3b-hcfa-adapter"
trainer.save_model(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

# Recommended on a free runtime: push the LoRA adapter to the Hub (needs a WRITE token)
# so you keep it if the session drops. The adapter is small (~tens of MB).
# model.push_to_hub("catochris/qwen2.5vl-3b-hcfa")
# processor.push_to_hub("catochris/qwen2.5vl-3b-hcfa")
print("saved adapter ->", ADAPTER_DIR)

## 10. Inference on the test split

Greedy decode, then pull the JSON object out of the model's text. We keep the raw
output too so the scorer can measure true JSON-validity.

In [ ]:
import json, re

model.config.use_cache = True
model.eval()


def predict(example, max_new_tokens=1536):
    msgs = to_messages(example, with_target=False)
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, _ = process_vision_info(msgs)
    inputs = processor(text=[text], images=imgs, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen, skip_special_tokens=True)[0].strip()


def extract_json(text):
    t = text.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?", "", t).strip().strip("`").strip()
    try:
        return json.loads(t)
    except Exception:
        m = re.search(r"\{.*\}", t, re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return None
    return None


# quick look at one prediction
raw = predict(ds["test"][0])
print(raw[:500])
print("parsed keys:", len(extract_json(raw) or {}))

## 11. Score with the repo's `hcfa_eval` harness

The HF dataset carries the flat-JSON `target`, so we rebuild a tiny `batch_dir`
(`{"logical": unflatten(target)}` per sample) + a split file, then call `summarize()`
— the same per-tier / per-class / CER / validity report the CLI prints. Each prediction
line includes `raw` so JSON-validity reflects the model's actual output.

In [ ]:
from pathlib import Path
from hcfa_eval.schema import unflatten
from hcfa_eval.scoring import summarize, format_summary, write_summary_csv

batch_dir = Path("eval_batch"); batch_dir.mkdir(exist_ok=True)
split_rows, pred_lines = [], []

test = ds["test"]
for i, ex in enumerate(test):
    sid = ex["sample_id"]
    # rebuild GT for the harness from the dataset target (flatten(unflatten(x)) == x for values)
    logical = unflatten(json.loads(ex["target"]))
    (batch_dir / f"{sid}.json").write_text(json.dumps({"logical": logical}), encoding="utf-8")
    split_rows.append({"sample_id": sid, "tier": ex["tier"], "json": f"{sid}.json"})

    raw = predict(ex)
    parsed = extract_json(raw)
    pred_lines.append(json.dumps({
        "sample_id": sid,
        "fields": parsed if isinstance(parsed, dict) else {},
        "raw": raw,
    }))
    if (i + 1) % 10 == 0:
        print(f"  predicted {i + 1}/{len(test)}")

Path("test_split.jsonl").write_text("\n".join(json.dumps(r) for r in split_rows), encoding="utf-8")
Path("preds.jsonl").write_text("\n".join(pred_lines), encoding="utf-8")

summary = summarize(Path("test_split.jsonl"), Path("preds.jsonl"), batch_dir, model="qwen2.5vl-3b-qlora")
print(format_summary(summary))
write_summary_csv(summary, Path("runs.csv"))
print("\nwrote runs.csv")

## 12. Where to go next

- **OOM on the T4?** Lower `MAX_PIXELS` (cell 5), keep batch size 1, or try
  **HuggingFaceTB/SmolVLM2-2.2B-Instruct** — smaller, lighter on 15 GB, faster cold
  start (a baseline the brief calls out).
- **Resolution sweep** — once it's stable, raise `MAX_PIXELS` and re-score; the brief
  flags resolution (not param count) as the main lever for the dense small-text fields.
  On the T4's capped resolution, expect the codes/money/NPI character accuracy to be the
  weakest spot.
- **Per-tier gap** — compare `pristine` vs `worst` in the summary; that's the
  degradation-robustness number that matters for production.
- **Overfitting** — only 404 synthetic examples; if val loss turns up, cut epochs or
  LoRA rank, and validate on a few real de-identified forms before trusting numbers.